Vidushi Mittal

M25MAC014

NLU-A2



## PROBLEM 2: CHARACTER-LEVEL NAME GENERATION USING RNN VARIANTS

Objective The objective of this assignment is to design and compare sequence models for
character-level name generation using recurrent neural architectures.

### TASK-0: DATASET

Generate 1000 Indian names using LLMs. Stores this as TrainingNames.txt.

In [1]:
!pip install Faker

In [2]:
from faker import Faker

def generate_indian_names(count=1000, filename="TrainingNames.txt"):
    # Initialize Faker with India locale
    fake = Faker('en_IN')

    names = set()
    print(f"Generating {count} unique Indian names...")

    # We use a set to ensure all 1000 names are unique
    while len(names) < count:
        # Extract just the first name and last name, removing prefixes (Mr./Dr.)
        full_name = fake.name()

        # Simple cleaning to remove titles like 'Dr.', 'Mr.', 'Mrs.'
        clean_name = ' '.join([word for word in full_name.split() if '.' not in word])

        # Basic validation: ensure it's not empty and contains only letters/spaces
        if clean_name and all(x.isalpha() or x.isspace() for x in clean_name):
            names.add(clean_name.strip())

    # Save to file
    with open(filename, "w", encoding="utf-8") as f:
        for name in sorted(list(names)):
            f.write(name + "\n")

    print(f"Successfully saved {len(names)} names to {filename}")

    print('Example 25 random names are:', list(names)[:25])
    return list(names)

# generating random names
training_names=generate_indian_names(1000)

Generating 1000 unique Indian names...
Successfully saved 1000 names to TrainingNames.txt
Example 25 random names are: ['Akshay Chada', 'Robert Bhakta', 'Lila Sarin', 'Jasmit Kata', 'Arunima Lala', 'Edhitha Shan', 'Lila Bath', 'Jagvi Gola', 'Warinder Naik', 'Gauri Rout', 'Ishaan Grewal', 'Vanya Kothari', 'Sudiksha Gupta', 'Ranbir Mangat', 'Siddharth Sur', 'Balendra Magar', 'Zinal Varma', 'Vansha Srivastava', 'Akshay Jha', 'Prisha Borah', 'Harsh Varma', 'Darsh Soni', 'Udyati Chand', 'Agastya Natarajan', 'Indira Ramesh']


TASK-1: MODEL IMPLEMENTATION
Implement from scratch and compare the following models:
1. Vanilla Recurrent Neural Network (RNN)
2. Bidirectional Long Short-Term Memory (BLSTM)
3. RNN with Basic Attention Mechanism

Data preposessing

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Load and preprocess
names = open('TrainingNames.txt').read().lower().splitlines()
all_chars = sorted(list(set(''.join(names)))) + ['<EOS>']
print(all_chars)
n_chars = len(all_chars)
print("Length of all_chars",n_chars)
# character mapping
char_to_idx = {c: i for i, c in enumerate(all_chars)}
print("\n char_to_idx: ",char_to_idx)
idx_to_char = {i: c for i, c in enumerate(all_chars)}
print("\n idx_to_char: ",idx_to_char)

def name_to_tensor(name):
    tensor = torch.zeros(len(name) + 1, 1, n_chars)
    for i, char in enumerate(name):
        tensor[i][0][char_to_idx[char]] = 1
    tensor[-1][0][char_to_idx['<EOS>']] = 1
    return tensor

def target_to_tensor(name):
    indices = [char_to_idx[c] for c in name]
    indices.append(char_to_idx['<EOS>'])
    return torch.LongTensor(indices)

[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '<EOS>']
Length of all_chars 28

 char_to_idx:  {' ': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '<EOS>': 27}

 idx_to_char:  {0: ' ', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 27: '<EOS>'}


In [4]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class NameDataset(Dataset):
    def __init__(self, names):
        self.names = names
    def __len__(self): return len(self.names)
    def __getitem__(self, idx):
        name = self.names[idx].lower()
        # input: 'amit' -> [a, m, i, t] | target: [m, i, t, <EOS>]
        input_seq = [char_to_idx[c] for c in name]
        target_seq = input_seq[1:] + [char_to_idx['<EOS>']]
        return torch.tensor(input_seq), torch.tensor(target_seq)

# Collate function to handle variable length names in a batch
def collate_fn(batch):
    inputs, targets = zip(*batch)
    inputs_pad = nn.utils.rnn.pad_sequence(inputs, batch_first=True, padding_value=char_to_idx['<EOS>'])
    targets_pad = nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=char_to_idx['<EOS>'])
    return inputs_pad, targets_pad

dataloader = DataLoader(NameDataset(training_names), batch_size=32, shuffle=True, collate_fn=collate_fn)

Model Architectures.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# vanilla RNN

class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size,num_layers=1):
        super(VanillaRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden=None):
        # x: [batch, seq_len]
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden


# Bidirectional LSTM

class BLSTMModel(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size, num_layers=1):
        super(BLSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_directions = 2 # For bidirectional
        self.embedding = nn.Embedding(vocab_size, hidden_size) # Add embedding layer
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=num_layers, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * self.num_directions, output_size) # *2 for bidirectional

    def forward(self, x, hidden=None):
        embedded = self.embedding(x) # Embed the input indices
        batch_size = embedded.size(0)

        if hidden is None:
            # Initialize h_0 and c_0 for LSTM
            h_0 = torch.zeros(self.num_layers * self.num_directions, batch_size, self.hidden_size)
            c_0 = torch.zeros(self.num_layers * self.num_directions, batch_size, self.hidden_size)
            hidden = (h_0, c_0)

        out, hidden = self.lstm(embedded, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        # Helper to explicitly initialize hidden state if needed
        h_0 = torch.zeros(self.num_layers * self.num_directions, batch_size, self.hidden_size)
        c_0 = torch.zeros(self.num_layers * self.num_directions, batch_size, self.hidden_size)
        return (h_0, c_0)


# RNN with basic attention

class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size,num_layers=1):
        super(AttentionRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

        # Simple dot-product attention
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)

        # Basic Attention: Compare current output with hidden state
        attn_weights = torch.bmm(output, hidden[-1].unsqueeze(2))
        attn_weights = torch.softmax(attn_weights, dim=1)
        context = torch.bmm(attn_weights.transpose(1, 2), output)
        context_expanded = context.expand(-1, output.size(1), -1)

        # 3. Concatenate and pass to Fully Connected layer
        combined = torch.cat((output, context_expanded), dim=2)
        out = self.fc(combined)

        return out, hidden

    def init_hidden(self, batch_size=1):
        return torch.zeros(1, batch_size, self.hidden_size)

In [6]:
# trainable prameter counter

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Usage
vanilla_rnn_model = VanillaRNN(n_chars, 128, n_chars)
BLSTM_model=BLSTMModel(n_chars, 128, n_chars)
AttentionRNN_model=AttentionRNN(n_chars, 128, n_chars)
print(f"Vanilla RNN Parameters: {count_parameters(vanilla_rnn_model)}")
print(f"BLSTM Parameters: {count_parameters(BLSTM_model)}")
print(f"Attention Parameters: {count_parameters(AttentionRNN_model)}")


Vanilla RNN Parameters: 40220
BLSTM Parameters: 274972
Attention Parameters: 126364


In [7]:
vanilla_rnn_model_1 = VanillaRNN(n_chars, 256, n_chars)
BLSTM_model_1=BLSTMModel(n_chars, 256, n_chars)
AttentionRNN_model_1=AttentionRNN(n_chars, 256, n_chars)
print(f"Vanilla RNN Parameters: {count_parameters(vanilla_rnn_model_1)}")
print(f"BLSTM Parameters: {count_parameters(BLSTM_model_1)}")
print(f"Attention Parameters: {count_parameters(AttentionRNN_model_1)}")

Vanilla RNN Parameters: 145948
BLSTM Parameters: 1074204
Attention Parameters: 482076


In [8]:
vanilla_rnn_model_2= VanillaRNN(n_chars, 64, n_chars)
BLSTM_model_2=BLSTMModel(n_chars, 64, n_chars)
AttentionRNN_model_2=AttentionRNN(n_chars, 64, n_chars)
print(f"Vanilla RNN Parameters: {count_parameters(vanilla_rnn_model_2)}")
print(f"BLSTM Parameters: {count_parameters(BLSTM_model_2)}")
print(f"Attention Parameters: {count_parameters(AttentionRNN_model_2)}")

Vanilla RNN Parameters: 11932
BLSTM Parameters: 71964
Attention Parameters: 34524


Training and sampling

In [9]:
def train_model(model, lr=0.002, epochs=30):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=char_to_idx['<EOS>'])

    model.train()
    for epoch in range(epochs):
        total_loss = 0 # Initialize total_loss for each epoch
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            output, _ = model(inputs)
            loss = criterion(output.transpose(1, 2), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Calculate average loss for this epoch
        final_loss = total_loss / len(dataloader)

    return final_loss # Return the loss of the very last epoch

def generate_name(model, max_len=15):
    model.eval()
    start_char = random.choice('abcdefghijklmnopqrstuvwxyz')
    name = start_char
    input_idx = torch.tensor([[char_to_idx[start_char]]])
    hidden = None

    for _ in range(max_len):
        output, hidden = model(input_idx, hidden)
        probs = F.softmax(output[0, -1], dim=0)
        top_idx = torch.multinomial(probs, 1).item()

        char = idx_to_char[top_idx]
        if char == '<EOS>': break
        name += char
        input_idx = torch.tensor([[top_idx]])
    return name

TASK-2: MODEL EVALUATION AND ANALYSIS.

In [10]:
def quantitative_evaluation(model, training_names, n_gen=300):
    generated = [generate_name(model) for _ in range(n_gen)]
    unique_gen = set(generated)
    train_set = set(training_names)

    novelty = len([n for n in unique_gen if n not in train_set]) / len(unique_gen)
    diversity = len(unique_gen) / n_gen

    return novelty * 100, diversity, generated[:5],generated

# --- EXECUTION AND COMPARISON ---

# Defining search space
hidden_sizes = [64, 128, 256]
learning_rates = [0.001, 0.005]
num_layers = [1, 2]

# Dictionary to track the best results for each model independently
best_results = {
    "Vanilla RNN": {"loss": float('inf'), "config": None, "nov": 0, "div": 0},
    "BLSTM": {"loss": float('inf'), "config": None, "nov": 0, "div": 0},
    "Attention RNN": {"loss": float('inf'), "config": None, "nov": 0, "div": 0}
}

print(f"\n{'Model':<15} | {'Hidden':<6} | {'LR':<5} | {'Layers':<6} | {'Loss':<6} | {'Nov %':<6} | {'Div':<5}")
print("-" * 65)

for hs in hidden_sizes:
    for lr in learning_rates:
        for layers in num_layers:

            # Initialize models with the correct loop variables (hs and layers)
            models = {
                "Vanilla RNN": VanillaRNN(n_chars, hs,n_chars, layers),
                "BLSTM": BLSTMModel(n_chars, hs,n_chars,layers),
                "Attention RNN": AttentionRNN(n_chars, hs,n_chars,layers)
            }

            for name, model in models.items():

                # 1. Train the model ONCE and capture the loss
                final_loss = train_model(model, lr=lr, epochs=20)

                # 2. Evaluate (n_gen reduced to 200 here to speed up the loop)
                nov, div, samples,names = quantitative_evaluation(model, training_names, n_gen=200)


                # Print current progress row
                print(f"{name:<15} | {hs:<6} | {lr:<5} | {layers:<6} | {final_loss:<6.4f} | {nov:<6.1f} | {div:<5.2f}")

                # 3. Check if this is the best configuration for THIS specific model
                if final_loss < best_results[name]["loss"]:
                    best_results[name]["loss"] = final_loss
                    best_results[name]["config"] = (hs, lr, layers)
                    best_results[name]["nov"] = nov
                    best_results[name]["div"] = div

# Print the final winning configurations for your report!
print("\n" + "="*50)
print("\n BEST CONFIGURATIONS FOUND ")
print("="*50)

for name, res in best_results.items():
    cfg = res["config"]
    if cfg:
        print(f"\n{name}:")
        print(f"  - Best Hyperparameters: Hidden={cfg[0]}, LR={cfg[1]}, Layers={cfg[2]}")
        print(f"  - Achieved Training Loss: {res['loss']:.4f}")
        print(f"  - Resulting Novelty: {res['nov']:.2f}% | Diversity: {res['div']:.2f}")


Model           | Hidden | LR    | Layers | Loss   | Nov %  | Div  
-----------------------------------------------------------------
Vanilla RNN     | 64     | 0.001 | 1      | 1.7741 | 100.0  | 1.00 
BLSTM           | 64     | 0.001 | 1      | 0.0266 | 100.0  | 0.96 
Attention RNN   | 64     | 0.001 | 1      | 1.4341 | 100.0  | 1.00 
Vanilla RNN     | 64     | 0.001 | 2      | 1.7866 | 100.0  | 1.00 
BLSTM           | 64     | 0.001 | 2      | 0.0064 | 100.0  | 0.94 
Attention RNN   | 64     | 0.001 | 2      | 1.3765 | 100.0  | 0.99 
Vanilla RNN     | 64     | 0.005 | 1      | 1.2945 | 100.0  | 1.00 
BLSTM           | 64     | 0.005 | 1      | 0.0016 | 100.0  | 0.80 
Attention RNN   | 64     | 0.005 | 1      | 0.5813 | 100.0  | 1.00 
Vanilla RNN     | 64     | 0.005 | 2      | 1.2754 | 100.0  | 1.00 
BLSTM           | 64     | 0.005 | 2      | 0.0005 | 100.0  | 0.69 
Attention RNN   | 64     | 0.005 | 2      | 0.5374 | 100.0  | 1.00 
Vanilla RNN     | 128    | 0.001 | 1      | 1.421